# Day 15 — 데이터를 받아 오고 출처(provenance)를 기록하기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-3/day-15.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-3/day-15.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 연습용 가상 데이터 만들기

실제 데이터를 건드리기 전에 같은 모양의 작은 가상 데이터로 연습합니다. 아래 코드는 12개 샘플 × 60개 가상 유전자(`Prac01`–`Prac60`)를 실제 RCC 파일과 같은 형식으로 저장합니다. 숫자는 모두 지어낸 것이므로 생물학적 의미가 없습니다. 노트북은 `notebooks/` 폴더 안에서 실행한다고 가정합니다(그래서 경로가 `../data/...`입니다).

In [ ]:
import gzip
from pathlib import Path
import numpy as np
import pandas as pd

rng = np.random.default_rng(536)
out = Path("../data/osdr_536_practice")
out.mkdir(parents=True, exist_ok=True)

sources = [f"F{i}" for i in range(1, 7)] + [f"GC{i}" for i in range(1, 7)]
conditions = ["Space Flight"] * 6 + ["Ground Control"] * 6
genes = [f"Prac{i:02d}" for i in range(1, 61)]

base = rng.uniform(4, 11, size=60)      # 유전자별 기본 발현량(log2)
effect = np.zeros(60)                   # 비행군에서의 가상 변화량(log2)
effect[0:6] = -0.9      # Prac01–06 감소
effect[20:25] = -0.7    # Prac21–25 감소
effect[30:34] = -0.8    # Prac31–34 감소
effect[40:42] = 0.8     # Prac41–42 증가

s_rows, a_rows = [], []
for n, (src, cond) in enumerate(zip(sources, conditions), start=1):
    scale = rng.uniform(0.6, 1.5)       # 샘플마다 다른 RNA 투입량 흉내
    log2 = base + effect * (cond == "Space Flight") + rng.normal(0, 0.35, 60)
    endo = rng.poisson(scale * 2 ** log2)
    hk = rng.poisson(scale * np.array([800, 300, 150, 3]))
    pos = rng.poisson([30000, 8000, 2000, 500, 120, 30])
    neg = rng.poisson(6, size=8)

    lines = ["<Header>", "FileVersion,1.7", "</Header>", "",
             "<Sample_Attributes>", f"ID,{src}-PRACTICE", "GeneRLF,PRACTICE_PANEL",
             "</Sample_Attributes>", "", "<Code_Summary>", "CodeClass,Name,Accession,Count"]
    lines += [f"Endogenous,{g},PRAC_{i:04d}.1,{c}" for i, (g, c) in enumerate(zip(genes, endo), 1)]
    lines += [f"Housekeeping,HK{i},PRAC_HK{i}.1,{c}" for i, c in enumerate(hk, 1)]
    lines += [f"Positive,POS_{'ABCDEF'[i]},ERCC_P{i}.1,{c}" for i, c in enumerate(pos)]
    lines += [f"Negative,NEG_{'ABCDEFGH'[i]},ERCC_N{i}.1,{c}" for i, c in enumerate(neg)]
    lines += ["</Code_Summary>", ""]

    fname = f"PRACTICE_microarray_P{n:02d}_{src}.RCC.gz"
    with gzip.open(out / fname, "wt") as f:
        f.write("\n".join(lines))
    s_rows.append({"Source Name": src, "Sample Name": f"P{n:02d}",
                   "Factor Value[Spaceflight]": cond})
    a_rows.append({"Sample Name": f"P{n:02d}", "Array Data File": fname})

pd.DataFrame(s_rows).to_csv(out / "s_PRACTICE.txt", sep="\t", index=False)
pd.DataFrame(a_rows).to_csv(out / "a_PRACTICE_ncounter.txt", sep="\t", index=False)

print("만든 파일 수:", len(list(out.glob("*"))))
with gzip.open(out / "PRACTICE_microarray_P01_F1.RCC.gz", "rt") as f:
    print("".join(f.readlines()[9:14]))

### 예시 2 — OSDR에 실제로 어떤 파일이 있는지 물어보기

웹 페이지를 눈으로 훑는 대신, OSDR이 제공하는 API에 "536번 study의 파일 목록을 달라"고 요청합니다. 인터넷 연결이 필요합니다.

In [ ]:
import json
import urllib.request

API = "https://osdr.nasa.gov/osdr/data/osd/files/536"
with urllib.request.urlopen(API) as r:
    info = json.load(r)

files = info["studies"]["OSD-536"]["study_files"]
print("파일 수:", len(files))
for f in files[:4]:
    print(f["category"], "|", f["file_name"], "|", f["file_size"], "bytes")

### 예시 3 — 체크섬 계산하기

In [ ]:
import hashlib

a = hashlib.sha256(b"Tnf,NM_013693.1,12").hexdigest()
b = hashlib.sha256(b"Tnf,NM_013693.1,13").hexdigest()
print(a[:16])
print(b[:16])
print("같은가?", a == b)

## 직접 해보기

1. `notebooks/` 폴더에 `02_osdr_qc_and_pca.ipynb`를 새로 만들고(이번 주 Day 15–19에 계속 씁니다), 예시 1의 코드를 실행해 `data/osdr_536_practice/`에 연습 파일이 14개 생겼는지 확인합니다.
2. 예시 2를 실행해 파일 수와 종류를 확인합니다. 브라우저에서 <https://osdr.nasa.gov/bio/repo/data/studies/OSD-536> 의 Files 탭도 열어 같은 목록이 보이는지 비교합니다.
3. 아래 코드로 실제 파일을 `data/osdr_536/`에 내려받고, 체크섬이 담긴 목록(`download_manifest.csv`)을 만들고, ISA 묶음을 풉니다. 전체 용량은 약 140KB입니다.

In [ ]:
import hashlib
import zipfile
from datetime import date
from pathlib import Path
import pandas as pd

raw = Path("../data/osdr_536")
raw.mkdir(parents=True, exist_ok=True)

rows = []
for f in files:                                   # 예시 2에서 만든 files
    target = raw / f["file_name"]
    if not target.exists():                       # 이미 받았으면 다시 받지 않음
        urllib.request.urlretrieve("https://osdr.nasa.gov" + f["remote_url"], target)
    rows.append({
        "file_name": f["file_name"],
        "category": f["category"],
        "bytes": target.stat().st_size,
        "sha256": hashlib.sha256(target.read_bytes()).hexdigest(),
        "downloaded": date.today().isoformat(),
    })

manifest = pd.DataFrame(rows)
manifest.to_csv(raw / "download_manifest.csv", index=False)

for z in raw.glob("*.zip"):                       # ISA 메타데이터 묶음 풀기
    with zipfile.ZipFile(z) as zf:
        zf.extractall(raw)
        print("압축 안의 파일:", zf.namelist())

print(manifest["category"].value_counts().to_string())
print("크기 일치:", (manifest["bytes"] == [f["file_size"] for f in files]).all())

4. 압축에서 나온 `s_...txt`(샘플 표)와 `a_...txt`(assay 표)를 열어 열 이름을 먼저 확인합니다. 파일 이름을 외워서 치지 말고 `glob`으로 찾습니다.

In [ ]:
s_path = next(raw.glob("s_*.txt"))
a_path = next(raw.glob("a_*.txt"))
s_table = pd.read_csv(s_path, sep="\t")
a_table = pd.read_csv(a_path, sep="\t")
print(s_path.name, s_table.shape)
print(a_table.columns.tolist())
print(s_table["Factor Value[Spaceflight]"].value_counts().to_string())

5. API의 메타데이터에서 버전과 인용 정보를 꺼냅니다. 어떤 키가 있는지 모를 때는 `.keys()`로 먼저 들여다봅니다.

In [ ]:
with urllib.request.urlopen("https://osdr.nasa.gov/osdr/data/osd/meta/536") as r:
    meta = json.load(r)["study"]["OSD-536"]

study = meta["studies"][0]
print("version:", meta["version"])
print("공개일:", study["publicReleaseDate"])
for c in study["comments"]:
    if c["name"] in ("DOI", "Data Source Accession", "Mission Name", "Project Identifier"):
        print(c["name"], "=", c["value"])
pub = study["publications"][0]
print(pub["authorList"], pub["title"], "doi:", pub["doi"])